In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
%matplotlib inline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('running on', device)

SAVE_FIGURES = False  # set to True to write the figures used on the slides to ./figures
def savefig(name):
    if SAVE_FIGURES:
        os.makedirs('figures', exist_ok=True)
        plt.savefig(f'figures/{name}.png', dpi=150, bbox_inches='tight')

In [ ]:
%%sh
# Download the data - you need to do this only once
wget --no-verbose --output-document=image_dog.jpg https://github.com/chrirupp/cv_course/raw/main/data/image_dog.jpg
wget --no-verbose --output-document=image_me.jpg https://github.com/chrirupp/cv_course/raw/main/data/image_me.jpg
wget --no-verbose --output-document=image_cc_1.jpg https://github.com/chrirupp/cv_course/raw/main/data/image_cc_1.jpg
wget --no-verbose --output-document=image_cc_3.jpg https://github.com/chrirupp/cv_course/raw/main/data/image_cc_3.jpg
wget --no-verbose --output-document=image.jpg https://github.com/chrirupp/cv_course/raw/main/data/image.jpg

# Lecture 17 - Representation and Self-Supervised Learning

What do self-supervised models learn? We colourise a grayscale photo (a pretext task), visualise DINOv2 patch features and use them to find correspondences between very different views, let a masked autoencoder fill in hidden patches, and compare representations on CIFAR-10 with a k-NN classifier and a linear probe. Finally, two vision-language models trained contrastively: CLIP (softmax over a batch) and SigLIP (sigmoid per pair), and CLIP *reads*.

Run this notebook on Google Colab (see the [readme](https://github.com/chrirupp/cv_course)). Cells that load large pretrained models are faster with a GPU runtime (*Edit > Notebook settings > T4 GPU*).

In [ ]:
def load_rgb(filename, max_side=None):
    image = cv2.cvtColor(cv2.imread(filename), cv2.COLOR_BGR2RGB)
    if max_side:
        s = max_side / max(image.shape[:2])
        image = cv2.resize(image, (0, 0), fx=s, fy=s, interpolation=cv2.INTER_AREA)
    return image

images = {name: load_rgb(f'{name}.jpg', 1280) for name in ['image_dog', 'image_me', 'image_cc_1', 'image_cc_3', 'image']}

## Colourisation as a pretext task

Colour is free supervision: convert a photo to the Lab colour space, give the network the lightness L and ask for the two colour channels (a, b). Zhang et al. (ECCV 2016) quantise the (a, b) plane into 313 bins and predict a *distribution* over them at every position, because colour is ambiguous: a grey shirt could have been red or blue, and regressing the mean of all plausible colours gives brownish grey. To show a single image they take the *annealed mean* (the distribution sharpened with a temperature T = 0.38, then averaged).

We rebuild their network (VGG-style convolutions, dilated in the middle) and load their released weights (123 MB). Sky, sea, grass and chalk come out right: their colours follow from texture and context. The jeans are not determined by the grey values, and the model picks a plausible but wrong colour.

In [ ]:
def conv_block(c_in, c_out, n, stride_last=1, dilation=1):
    layers = []
    for k in range(n):
        layers += [nn.Conv2d(c_in if k == 0 else c_out, c_out, 3, stride_last if k == n - 1 else 1, dilation, dilation=dilation),
                   nn.ReLU(True)]
    return layers + [nn.BatchNorm2d(c_out)]

class Colorizer(nn.Module):
    # the ECCV 2016 colourisation network (github.com/richzhang/colorization), layer names as in their weights
    def __init__(self):
        super().__init__()
        self.model1 = nn.Sequential(*conv_block(1, 64, 2, stride_last=2))
        self.model2 = nn.Sequential(*conv_block(64, 128, 2, stride_last=2))
        self.model3 = nn.Sequential(*conv_block(128, 256, 3, stride_last=2))
        self.model4 = nn.Sequential(*conv_block(256, 512, 3))
        self.model5 = nn.Sequential(*conv_block(512, 512, 3, dilation=2))
        self.model6 = nn.Sequential(*conv_block(512, 512, 3, dilation=2))
        self.model7 = nn.Sequential(*conv_block(512, 512, 3))
        self.model8 = nn.Sequential(nn.ConvTranspose2d(512, 256, 4, 2, 1), nn.ReLU(True),
                                    nn.Conv2d(256, 256, 3, 1, 1), nn.ReLU(True), nn.Conv2d(256, 256, 3, 1, 1), nn.ReLU(True),
                                    nn.Conv2d(256, 313, 1))
        self.model_out = nn.Conv2d(313, 2, 1, bias=False)   # weights: the (a, b) centres of the 313 bins, divided by 110

    def forward(self, L):
        x = (L - 50) / 100
        for block in [self.model1, self.model2, self.model3, self.model4, self.model5, self.model6, self.model7, self.model8]:
            x = block(x)
        return x   # logits over the 313 colour bins, at 1/4 of the input resolution

colorizer = Colorizer()
colorizer.load_state_dict(torch.hub.load_state_dict_from_url(
    'https://colorizers.s3.us-east-2.amazonaws.com/colorization_release_v2-9b330a0b.pth', map_location='cpu', check_hash=True))
colorizer = colorizer.to(device).eval()
ab_bins = colorizer.model_out.weight[:, :, 0, 0].T.detach().cpu().numpy() * 110   # (313, 2)

def colorize(rgb, T=0.38):
    L = cv2.cvtColor(rgb.astype(np.float32) / 255, cv2.COLOR_RGB2LAB)[..., 0]     # lightness 0..100
    with torch.no_grad():
        logits = colorizer(torch.from_numpy(cv2.resize(L, (256, 256), interpolation=cv2.INTER_AREA))[None, None].to(device))[0]
    p = (logits / T).softmax(0).cpu().numpy()        # annealed: softmax(log p / T) = softmax(logits / T)
    ab = np.einsum('khw,kc->hwc', p, ab_bins)         # the mean of the sharpened distribution
    ab = cv2.resize(ab, (L.shape[1], L.shape[0]), interpolation=cv2.INTER_LINEAR)
    return L, np.clip(cv2.cvtColor(np.dstack([L, ab]).astype(np.float32), cv2.COLOR_LAB2RGB), 0, 1)

photo = load_rgb('image_me.jpg', 640)
L, colourised = colorize(photo)
y0, y1, x0, x1 = 170, 400, 300, 470                 # the person
fig, axs = plt.subplots(2, 3, figsize=(18, 10), gridspec_kw={'height_ratios': [1, 1.6]})
for col, (im, title) in enumerate([(L / 100, 'grayscale input (L)'), (colourised, 'prediction'), (photo / 255, 'ground truth')]):
    axs[0, col].imshow(im, cmap='gray' if im.ndim == 2 else None, vmin=0, vmax=1)
    axs[0, col].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, color='yellow', linewidth=2))
    axs[0, col].set_title(title, fontsize=16)
    axs[1, col].imshow(im[y0:y1, x0:x1], cmap='gray' if im.ndim == 2 else None, vmin=0, vmax=1)
    for ax in axs[:, col]:
        ax.axis('off')
plt.tight_layout()
savefig('ssl_color_ambiguity')
plt.show()

## DINOv2 patch features

DINOv2 is a ViT trained without labels (self-distillation). Every 14x14 patch gets a 768-dimensional feature. For each image, we project the patch features onto their first three principal components and show them as RGB. Without ever seeing a label or a mask, the features separate objects from the background and group their parts.

In [ ]:
from transformers import AutoModel
dino = AutoModel.from_pretrained('facebook/dinov2-base').to(device).eval()
mean, std = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])

def dino_features(image, patches_long_side=64):
    # resize so that both sides are multiples of the patch size (14)
    h, w = image.shape[:2]
    s = patches_long_side * 14 / max(h, w)
    ph, pw = int(round(h * s / 14)), int(round(w * s / 14))
    x = cv2.resize(image, (pw * 14, ph * 14), interpolation=cv2.INTER_AREA) / 255.0
    x = torch.from_numpy((x - mean) / std).permute(2, 0, 1)[None].float().to(device)
    with torch.no_grad():
        tokens = dino(pixel_values=x).last_hidden_state[0, 1:]     # drop the class token
    return F.normalize(tokens, dim=-1).cpu().numpy().reshape(ph, pw, -1)

features = {name: dino_features(im) for name, im in images.items()}

def pca_rgb(f):
    patches = f.reshape(-1, f.shape[-1])
    projected = PCA(n_components=3).fit_transform(patches)
    lo, hi = np.percentile(projected, 1, axis=0), np.percentile(projected, 99, axis=0)
    return np.clip((projected - lo) / (hi - lo), 0, 1).reshape(*f.shape[:2], 3)

names = ['image_dog', 'image_me', 'image_cc_1', 'image']
fig, axs = plt.subplots(2, len(names), figsize=(6 * len(names), 8))
for col, name in enumerate(names):
    f = features[name]
    rgb = pca_rgb(f)
    axs[0, col].imshow(images[name])
    axs[1, col].imshow(rgb, interpolation='nearest')
    axs[1, col].set_title(f'{f.shape[0]}x{f.shape[1]} patch features', fontsize=18)
    for ax in axs[:, col]:
        ax.axis('off')
fig.suptitle('DINOv2 (ViT-B/14) patch features: first three principal components (per image) as RGB', fontsize=20)
savefig('ssl_dino_pca')
plt.show()

## Correspondences from self-supervised features

The two photos below were taken from very different angles: SIFT with the ratio test finds only a few reliable matches (Lecture 5). DINOv2 features are much more invariant. We match patches by *mutual nearest neighbours* in feature space (cosine similarity) and, as for SIFT, keep the most *distinctive* matches: those whose best match is much more similar than the second best (this removes the sky, where every patch looks like every other). The matches are coarse (one per 14x14 patch, after resizing).

In [ ]:
fa, fb = features['image_cc_1'], features['image_cc_3']
A, B = fa.reshape(-1, fa.shape[-1]), fb.reshape(-1, fb.shape[-1])
similarity = A @ B.T
nn_ab, nn_ba = similarity.argmax(1), similarity.argmax(0)
mutual = np.where(nn_ba[nn_ab] == np.arange(len(A)))[0]
top2 = np.sort(similarity, axis=1)[:, -2:]
distinctiveness = top2[:, 1] - top2[:, 0]          # margin between the best and the second-best match
best = mutual[np.argsort(-distinctiveness[mutual])[:60]]
print(f'{len(mutual)} mutual nearest neighbours, showing the 60 most distinctive')

def patch_center(index, f, image):
    r, c = np.unravel_index(index, f.shape[:2])
    return np.array([(c + 0.5) * image.shape[1] / f.shape[1], (r + 0.5) * image.shape[0] / f.shape[0]])

# for comparison: SIFT with the ratio test
sift = cv2.SIFT_create()
ga, gb = [cv2.cvtColor(images[n], cv2.COLOR_RGB2GRAY) for n in ['image_cc_1', 'image_cc_3']]
ka, da = sift.detectAndCompute(ga, None)
kb, db = sift.detectAndCompute(gb, None)
sift_matches = [m for m, n in cv2.BFMatcher().knnMatch(da, db, k=2) if m.distance < 0.7 * n.distance]
print(f'SIFT: {len(sift_matches)} matches pass the ratio test (0.7)')

ia, ib = images['image_cc_1'], images['image_cc_3']
canvas = np.concatenate([ia, ib], axis=1)
fig, ax = plt.subplots(figsize=(20, 7))
ax.imshow(canvas)
colors = plt.cm.hsv(np.linspace(0, 1, len(best), endpoint=False))
for i, color in zip(best, colors):
    pa, pb = patch_center(i, fa, ia), patch_center(nn_ab[i], fb, ib) + [ia.shape[1], 0]
    ax.plot([pa[0], pb[0]], [pa[1], pb[1]], '-', color=color, linewidth=1)
    ax.scatter(*np.stack([pa, pb]).T, color=color, s=15)
ax.set_title(f'DINOv2 mutual nearest neighbours (60 most distinctive); SIFT + ratio test finds {len(sift_matches)} matches')
ax.axis('off')
savefig('ssl_dino_correspondences')
plt.show()

## Masked autoencoders (MAE)

MAE (He et al., CVPR 2022) hides a random 75% of the 16x16 patches. The encoder (a ViT-B) sees only the visible 25%, which makes pre-training cheap; a small decoder gets the encoded visible patches plus a learned *mask token* at every hidden position and predicts the hidden pixels. The loss is the mean squared error on the hidden patches only.

The pretrained model was never trained on these photos. Where the content is predictable (the repeated windows, the snow), the reconstruction is good; where it is not (the dog's face, the exact texture), it is blurry: with a squared error, the best guess under uncertainty is the mean (as for colourisation).

In [ ]:
from transformers import ViTMAEForPreTraining
mae = ViTMAEForPreTraining.from_pretrained('facebook/vit-mae-base').to(device).eval()

def center_square(image, size=224):
    h, w = image.shape[:2]
    s = min(h, w)
    crop = image[(h - s) // 2:(h + s) // 2, (w - s) // 2:(w + s) // 2]
    return cv2.resize(crop, (size, size), interpolation=cv2.INTER_AREA) / 255.0

fig, axs = plt.subplots(2, 3, figsize=(15, 10))
for row, name in enumerate(['image_dog', 'image_cc_1']):
    img = center_square(images[name])
    x = torch.from_numpy((img - mean) / std).permute(2, 0, 1)[None].float().to(device)
    noise = torch.rand(1, 196, generator=torch.Generator().manual_seed(0)).to(device)   # fixes which patches are hidden
    with torch.no_grad():
        out = mae(pixel_values=x, noise=noise)
    hidden = np.kron(out.mask[0].cpu().numpy().reshape(14, 14), np.ones((16, 16)))[..., None]   # 1 = hidden
    predicted = np.clip(mae.unpatchify(out.logits)[0].permute(1, 2, 0).cpu().numpy() * std + mean, 0, 1)
    visible = img * (1 - hidden) + 0.5 * hidden
    reconstruction = img * (1 - hidden) + predicted * hidden    # the loss only covers the hidden patches
    print(f'{name}: {hidden.mean():.0%} of the pixels hidden, loss (on ImageNet-normalised pixels) {out.loss.item():.3f}')
    for col, (im, title) in enumerate([(visible, 'encoder input: 25% of the patches'),
                                       (reconstruction, 'reconstruction'), (img, 'original')]):
        axs[row, col].imshow(im)
        axs[row, col].set_title(title, fontsize=16)
        axs[row, col].axis('off')
plt.tight_layout()
savefig('ssl_mae')
plt.show()

## Evaluating representations: k-NN and linear probes

A self-supervised model is judged by how useful its features are for a task it was not trained on. We freeze each model, embed 5000 CIFAR-10 training and 1000 test images (upsampled to 224x224) and classify the test images in two ways:

* **k-NN:** the labels of the k = 20 most similar training images (cosine similarity) vote, weighted by $\exp(s / 0.07)$, as in the DINO paper. No training at all.
* **Linear probe:** a logistic regression on the frozen features.

The models: a ViT with random weights, a ResNet-50 trained with ImageNet labels, MAE, CLIP and DINOv2. This takes a few minutes on a T4. (Practical 4 trains a probe on its own RotNet.)

In [ ]:
cifar_train = torchvision.datasets.CIFAR10('data', train=True, download=True)
cifar_test = torchvision.datasets.CIFAR10('data', train=False, download=True)
classes = cifar_train.classes
X_train, y_train = torch.tensor(cifar_train.data[:5000]), torch.tensor(cifar_train.targets[:5000])
X_test, y_test = torch.tensor(cifar_test.data[:1000]), torch.tensor(cifar_test.targets[:1000])

from transformers import CLIPModel, ViTMAEModel, Dinov2Config, Dinov2Model
torch.manual_seed(0)
random_vit = Dinov2Model(Dinov2Config.from_pretrained('facebook/dinov2-base')).to(device).eval()   # same architecture, random weights
resnet = torchvision.models.resnet50(weights='IMAGENET1K_V2').to(device).eval()
resnet.fc = nn.Identity()                                   # the 2048-d features before the classifier
mae_encoder = ViTMAEModel.from_pretrained('facebook/vit-mae-base', mask_ratio=0.0).to(device).eval()   # nothing hidden
clip_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch16').to(device).eval()

imagenet_norm = (torch.tensor([0.485, 0.456, 0.406]), torch.tensor([0.229, 0.224, 0.225]))
clip_norm = (torch.tensor([0.4815, 0.4578, 0.4082]), torch.tensor([0.2686, 0.2613, 0.2758]))

def clip_image_features(x):
    f = clip_model.get_image_features(pixel_values=x)
    return f if torch.is_tensor(f) else f.pooler_output    # newer transformers return an output object

models = {   # name: (frozen feature extractor, input normalisation)
    'random ViT-B': (lambda x: random_vit(pixel_values=x).pooler_output, imagenet_norm),
    'supervised\nResNet-50': (resnet, imagenet_norm),
    'MAE ViT-B': (lambda x: mae_encoder(pixel_values=x).last_hidden_state[:, 1:].mean(1), imagenet_norm),
    'CLIP ViT-B/16': (clip_image_features, clip_norm),
    'DINOv2 ViT-B/14': (lambda x: dino(pixel_values=x).pooler_output, imagenet_norm),
}

def embed(extract, norm, X, batch_size=250):
    m, s = norm[0].view(1, 3, 1, 1), norm[1].view(1, 3, 1, 1)
    out = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            x = X[i:i + batch_size].permute(0, 3, 1, 2).float() / 255
            x = F.interpolate(x, size=224, mode='bicubic', align_corners=False).clamp(0, 1)
            out.append(F.normalize(extract(((x - m) / s).to(device)).float(), dim=-1).cpu())
    return torch.cat(out)

def knn_classify(f_train, f_test, k=20, T=0.07):
    s, idx = (f_test @ f_train.T).topk(k, dim=1)
    votes = torch.zeros(len(f_test), 10).scatter_add_(1, y_train[idx], (s / T).exp())
    return votes.argmax(1), idx

results, neighbours = {}, {}
for name, (extract, norm) in tqdm(models.items()):
    f_train, f_test = embed(extract, norm, X_train), embed(extract, norm, X_test)
    pred, idx = knn_classify(f_train, f_test)
    probe = LogisticRegression(max_iter=2000, C=1.0).fit(f_train.numpy(), y_train.numpy())
    results[name] = ((pred == y_test).float().mean().item(), probe.score(f_test.numpy(), y_test.numpy()))
    neighbours[name] = idx[:, :5]
    print(f'{name.replace(chr(10), " "):18s} k-NN {results[name][0]:.1%}   linear probe {results[name][1]:.1%}')

In [ ]:
# accuracy of the two evaluations
names = list(results)
fig, ax = plt.subplots(figsize=(14, 5.2))
w = 0.38
for j, (label, color) in enumerate([('k-NN (k = 20)', 'tab:blue'), ('linear probe', 'tab:orange')]):
    acc = [results[n][j] for n in names]
    bars = ax.bar(np.arange(len(names)) + (j - 0.5) * w, acc, w, label=label, color=color)
    for b, a in zip(bars, acc):
        ax.text(b.get_x() + b.get_width() / 2, a + 0.01, f'{a:.0%}', ha='center', fontsize=15)
ax.axhline(0.1, color='gray', linestyle=':', label='chance')
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, fontsize=15)
ax.set_ylim(0, 1.08)
ax.set_ylabel('CIFAR-10 test accuracy', fontsize=15)
ax.legend(loc='upper left', fontsize=14)
for side in ['top', 'right']:
    ax.spines[side].set_visible(False)
plt.tight_layout()
savefig('ssl_knn_accuracy')
plt.show()

# the four nearest training images of a few test images
queries = [3, 12, 25, 15]
K = 4
shown = ['random ViT-B', 'MAE ViT-B', 'DINOv2 ViT-B/14']
fig, axs = plt.subplots(len(queries), 1 + len(shown) * K, figsize=(20, 1.62 * len(queries)),
                        gridspec_kw={'width_ratios': [1.2] + [1] * (len(shown) * K)})
for r, q in enumerate(queries):
    axs[r, 0].imshow(X_test[q].numpy())
    axs[r, 0].set_title(f'query: {classes[y_test[q]]}' if r == 0 else classes[y_test[q]], fontsize=15)
    for m, name in enumerate(shown):
        for k, i in enumerate(neighbours[name][q][:K]):
            ax = axs[r, 1 + m * K + k]
            ax.imshow(X_train[i].numpy())
            ok = y_train[i] == y_test[q]
            for side in ax.spines.values():
                side.set_edgecolor('tab:green' if ok else 'tab:red')
                side.set_linewidth(3)
            if r == 0 and k == 1:
                ax.set_title(name, fontsize=17, x=1.05)
for ax in axs.flat:
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
savefig('ssl_knn_neighbours')
plt.show()

## CLIP vs. SigLIP

Both models embed images and texts into a joint space and score image-text pairs by cosine similarity. They differ in the training loss:

* **CLIP** uses a *softmax* over all texts in the batch: the probabilities of all captions for an image sum to one, so captions compete.
* **SigLIP** uses a *sigmoid* per image-text pair: every caption gets an independent probability of being a match.

For an image that fits several captions, CLIP has to spread or pick; SigLIP can say yes to all of them.

In [ ]:
from transformers import CLIPModel, CLIPProcessor, AutoProcessor
clip = CLIPModel.from_pretrained('openai/clip-vit-base-patch16').to(device).eval()
clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch16')
siglip = AutoModel.from_pretrained('google/siglip-base-patch16-224').to(device).eval()
siglip_processor = AutoProcessor.from_pretrained('google/siglip-base-patch16-224')

def clip_probs(image, texts):
    inputs = clip_processor(text=texts, images=image, return_tensors='pt', padding=True).to(device)
    with torch.no_grad():
        return clip(**inputs).logits_per_image.softmax(-1)[0].cpu().numpy()

def siglip_probs(image, texts):
    inputs = siglip_processor(text=texts, images=image, return_tensors='pt', padding='max_length', max_length=64).to(device)
    with torch.no_grad():
        return torch.sigmoid(siglip(**inputs).logits_per_image)[0].cpu().numpy()

captions = ['a photo of a dog.', 'a photo of a dog in the snow.', 'a photo of a dog next to a lantern.',
            'a photo of snow.', 'a photo of a lantern.', 'a photo of a cat.', 'a photo of a beach.']
image = images['image_dog']
p_clip, p_siglip = clip_probs(image, captions), siglip_probs(image, captions)

fig, axs = plt.subplots(1, 3, figsize=(21, 8), gridspec_kw={'width_ratios': [1.15, 1.25, 1.25]})
axs[0].imshow(image)
axs[0].axis('off')
for ax, p, title in [(axs[1], p_clip, f'CLIP: softmax over captions (sums to {p_clip.sum():.2f})'),
                     (axs[2], p_siglip, f'SigLIP: independent sigmoid per caption (sums to {p_siglip.sum():.2f})')]:
    ax.barh(range(len(captions)), p, color='tab:blue')
    ax.set_yticks(range(len(captions)))
    ax.set_yticklabels(captions, fontsize=19)
    ax.invert_yaxis()
    ax.set_xlim(0, 1.15)
    ax.set_xticks([0, 0.5, 1])
    ax.tick_params(axis='x', labelsize=17)
    ax.set_title(title, fontsize=19)
    for i, v in enumerate(p):
        ax.text(v + 0.01, i, f'{v:.2f}', va='center', fontsize=19)
plt.tight_layout(w_pad=4)
savefig('ssl_clip_vs_siglip')
plt.show()

## Models learn to read

CLIP is trained on images with their alt-text from the web, where text in the image often matches the caption. So it learned to read, and text in an image can override what it shows: a *typographic attack*. We put a small sticker with a word next to the dog and classify the image zero-shot.

In [ ]:
def sticker(image, text):
    image = image.copy()
    h, w = image.shape[:2]
    cv2.rectangle(image, (int(0.6 * w), int(0.55 * h)), (int(0.95 * w), int(0.8 * h)), (255, 255, 255), -1)
    cv2.putText(image, text, (int(0.62 * w), int(0.73 * h)), cv2.FONT_HERSHEY_DUPLEX, 4 * w / 1280, (0, 0, 0), int(8 * w / 1280))
    return image

labels = ['dog', 'cat', 'fox', 'wolf', 'lantern', 'car']
prompts = [f'a photo of a {l}.' for l in labels]
fig, axs = plt.subplots(2, 4, figsize=(24, 9), gridspec_kw={'height_ratios': [1.6, 1]})
for col, text in enumerate([None, 'CAT', 'FOX', 'iPod']):
    attacked = image if text is None else sticker(image, text)
    p = clip_probs(attacked, prompts)
    axs[0, col].imshow(attacked)
    axs[0, col].set_title('original' if text is None else f'with a "{text}" sticker', fontsize=20)
    axs[0, col].axis('off')
    axs[1, col].bar(labels, p, color=['tab:red' if i == p.argmax() else 'tab:blue' for i in range(len(labels))])
    axs[1, col].set_ylim(0, 1)
    axs[1, col].tick_params(axis='y', labelsize=16)
    axs[1, col].tick_params(axis='x', labelsize=14)
    axs[1, col].set_title(f'CLIP zero-shot: {labels[p.argmax()]} ({p.max():.2f})', fontsize=20)
savefig('ssl_typographic_attack')
plt.show()